# Stage Four: The Economics of Contacting Customers

This notebook is a record of the code for Stage Four. What each result showed, and the reasoning behind each decision, are set out in `md/04_economics_writeup.md`.

---

## 1. Combining the two parts

This joins the model's scores to the customer records, so that every customer carries both a probability of leaving and a calculated value.

The join is done in the database rather than assembled in Python, which keeps to the split of work followed throughout the project.

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect("../telco.duckdb")

econ = con.execute("""
    SELECT s.customer_id,
           s.clv,
           s.value_decile,
           s.churn_probability,
           s.churn_value,
           f.contract,
           f.tenure_band,
           f.monthly_charges
    FROM customer_scores s
    JOIN vw_customer_features f ON s.customer_id = f.customer_id
""").df()

print(econ.shape)
econ.head()

(7043, 8)


,customer_id,clv,value_decile,churn_probability,churn_value,contract,tenure_band,monthly_charges
0,3668-QPYBK,193.86,8,0.551139,1,Month-to-month,New,53.85
1,9237-HQITU,254.52,7,0.838677,1,Month-to-month,New,70.70
2,9305-CDSKC,358.74,4,0.893799,1,Month-to-month,New,99.65
3,7892-POOKP,377.28,4,0.808888,1,Month-to-month,Established,104.80
4,0280-XJGEX,373.32,4,0.591416,1,Month-to-month,Loyal,103.70


## 2. Applying the decision rule

This runs `05_economics.sql`, which creates `vw_intervention_economics`, then groups customers by the decision the rule reaches and totals the value recovered, the cost and the net figure for each group.

The rule is a single calculation applied to every customer:

> value of contacting a customer = chance of leaving × chance of keeping them × customer value − cost of contact

Where the answer is positive, contact is worth it. Where it is negative, contacting the customer destroys value, and that holds even when the customer really was about to leave.

Two figures in that calculation are not in the dataset. Both are assumptions, and both are written into `sql/05_economics.sql` next to the calculation that uses them. The cost of contact is set at R60 per customer, made up of R20 for an automated message and R40 for the offer itself. The chance of keeping a customer who would otherwise leave is set at 30 percent.

The cost is charged for every customer contacted, including those who were never leaving. That is what makes the rule work: it builds the model's error rate into the arithmetic rather than leaving it as a separate problem.

In [2]:
con.execute(open("../sql/05_economics.sql").read())

con.execute("""
    SELECT decision,
           COUNT(*) AS customers,
           ROUND(SUM(expected_value_saved),2) AS value_saved,
           ROUND(SUM(intervention_cost),2) AS total_cost,
           ROUND(SUM(net_value),2) AS net
    FROM vw_intervention_economics
    GROUP BY 1
""").df()

,decision,customers,value_saved,total_cost,net
0,Do not intervene,4825,105819.35,289500.0,-183680.65
1,Intervene,2218,172742.37,133080.0,39662.37


## 3. Finding the ceiling on recoverable value

This asks what can be recovered from a single customer, printing the smallest, average and largest amounts along with the ninetieth and ninety-ninth percentiles.

The rule was first built on a cost of R340 per customer, covering a telephone call from a member of staff and a larger offer held for six months. Under that assumption no customer at all qualified for contact. This query is what was run to find out why, and it is kept here because the answer decides whether any given cost assumption can work before a campaign is designed around it.

In [3]:
con.execute("""
    SELECT ROUND(MIN(expected_value_saved),2)  AS min_saved,
           ROUND(AVG(expected_value_saved),2)  AS avg_saved,
           ROUND(MAX(expected_value_saved),2)  AS max_saved,
           ROUND(QUANTILE_CONT(expected_value_saved, 0.90),2) AS p90,
           ROUND(QUANTILE_CONT(expected_value_saved, 0.99),2) AS p99
    FROM vw_intervention_economics
""").df()

,min_saved,avg_saved,max_saved,p90,p99
0,0.56,39.55,163.09,82.28,109.24


## 4. Spreading the decision across the value groups

This applies the rule within each of the ten value groups set up at Stage Two, from the most valuable customers to the least.

For each group it counts how many customers qualify for contact, totals the net figure for those that do, and then totals the net figure across every customer in the group. The last column is what the group would return if everyone in it were contacted without applying the rule, which is what makes the two approaches directly comparable.

In [4]:
con.execute("""
    SELECT value_decile,
           COUNT(*) AS customers,
           SUM(CASE WHEN decision = 'Intervene' THEN 1 ELSE 0 END) AS intervene,
           ROUND(SUM(CASE WHEN decision = 'Intervene' THEN net_value ELSE 0 END),2) AS net_if_targeted,
           ROUND(SUM(net_value),2) AS net_if_all_contacted
    FROM vw_intervention_economics
    GROUP BY 1 ORDER BY 1
""").df()

,value_decile,customers,intervene,net_if_targeted,net_if_all_contacted
0,1,705,83.0,1599.74,-17760.85
1,2,705,349.0,10261.63,-2598.32
2,3,705,138.0,2642.34,-17693.18
3,4,704,589.0,13506.37,11195.62
4,5,704,532.0,8206.15,5215.91
5,6,704,363.0,2974.06,-9179.05
6,7,704,164.0,472.08,-19150.76
7,8,704,0.0,0.00,-28863.49
8,9,704,0.0,0.00,-30433.87
9,10,704,0.0,0.00,-34750.29


## 5. Testing the assumption on the retention success rate

This runs `06_sensitivity.sql`, which creates `vw_sensitivity`, and displays the result.

The 30 percent success rate used above is the least certain figure in the project, and every result rests on it. This file tests what happens when it is wrong. It cross joins the scored customers against a range of success rates from 10 percent to 50 percent, holds the cost at R60 and leaves the model, the probabilities and the customer values untouched, then applies the rule again at each rate.

The row for 30 percent repeats the figures produced earlier in this notebook, which is the check that the two views agree with each other.

In [5]:
with open('../sql/06_sensitivity.sql', 'r') as f:
    con.execute(f.read())

sensitivity = con.execute("SELECT * FROM vw_sensitivity").df()
sensitivity

,success_rate,customers_contacted,pct_of_base,net_return_targeted,net_return_contact_everyone,total_cost,return_on_spend_pct
0,0.10,0,0.0,NaN,-329726.07,0.0,NaN
1,0.15,35,0.5,279.66,-283299.10,2100.0,13.3
2,0.20,361,5.1,2819.67,-236872.13,21660.0,13.0
3,0.25,1333,18.9,15395.12,-190445.16,79980.0,19.2
4,0.30,2218,31.5,39662.16,-144018.20,133080.0,29.8
5,0.35,2592,36.8,70514.16,-97591.23,155520.0,45.3
6,0.40,2857,40.6,104010.32,-51164.26,171420.0,60.7
7,0.50,3315,47.1,176163.62,41689.67,198900.0,88.6


---

Stage Four ends here, and with it the analysis. The findings, including the split between customers worth contacting and customers worth leaving alone, the comparison against contacting everyone, the pattern across the value groups and the results of the sensitivity test, are set out in `md/04_economics_writeup.md`.

**Remaining work:** export the scored and classified customer base in SQL, and build the reporting layer on top of it.